# 20 — Single Game Backtest

End-to-end backtest on one game: align trades with game state, compute edge at each trade, simulate PnL.

In [ ]:
import sys
sys.path.insert(0, '/Users/chriswang/Desktop/prediction-market-exploration/nba-edge')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from nba_edge.data.s3_reader import discover_trade_dates, load_trades, load_boxscores_for_game, discover_game_ids
from nba_edge.data.ticker_parser import parse_ticker, is_game_winner
from nba_edge.backtest.engine import BacktestEngine
from nba_edge.backtest.metrics import simulate_pnl, brier_score
from nba_edge.models.analytical import AnalyticalWinProb

In [ ]:
# Load trades
dates = discover_trade_dates(min_date='2026-04-21')
df = load_trades(dates)

# Find most-traded GAME ticker
game_tickers = [t for t in df['market_ticker'].unique().to_list() if is_game_winner(t)]
ticker_volumes = [(t, df.filter(pl.col('market_ticker') == t).height) for t in game_tickers]
ticker_volumes.sort(key=lambda x: -x[1])
chosen = ticker_volumes[0][0]
print(f'Backtesting: {chosen} ({ticker_volumes[0][1]} trades)')

In [ ]:
# Load boxscores
from nba_edge.data.game_alignment import match_ticker_to_game

parsed = parse_ticker(chosen)
available = discover_game_ids(parsed.game_date)
matched_id = match_ticker_to_game(chosen, available)
assert matched_id, f"No game match for {chosen}! Available: {[(g['home_team'], g['away_team']) for g in available]}"

snapshots = load_boxscores_for_game(matched_id, parsed.game_date)
print(f"Game: {matched_id}")
print(f"Snapshots: {len(snapshots)}")
print(f"Final: {snapshots[-1]['home_score']}-{snapshots[-1]['away_score']}")

In [ ]:
# Run backtest
engine = BacktestEngine(model=AnalyticalWinProb(variance_rate=0.44))
ticker_trades = df.filter(pl.col('market_ticker') == chosen)

result = engine.run_game(ticker_trades, snapshots, chosen)
print(f'Aligned trades: {len(result.trades)}')
print(f'Outcome (yes won): {result.outcome_yes}')
print(f'YES team: {result.yes_team}, Home: {result.home_team}, Away: {result.away_team}')

In [ ]:
# Edge distribution
edges = [t.edge * 100 for t in result.trades]  # in cents

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(edges, bins=50, edgecolor='none', alpha=0.7)
axes[0].axvline(0, color='black', linestyle='-')
axes[0].axvline(5, color='red', linestyle='--', alpha=0.5)
axes[0].axvline(-5, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Edge (cents)')
axes[0].set_ylabel('Count')
axes[0].set_title('Edge Distribution')

# Edge over time (by seconds remaining)
secs_remaining = [t.seconds_remaining for t in result.trades]
axes[1].scatter(secs_remaining, edges, s=2, alpha=0.3)
axes[1].axhline(0, color='black', linestyle='-')
axes[1].set_xlabel('Seconds Remaining')
axes[1].set_ylabel('Edge (cents)')
axes[1].set_title('Edge vs Time Remaining')
axes[1].invert_xaxis()

plt.tight_layout()
plt.show()

print(f'Mean edge: {np.mean(edges):.2f}c')
print(f'|edge| > 5c: {sum(1 for e in edges if abs(e) > 5)} trades ({sum(1 for e in edges if abs(e) > 5)/len(edges)*100:.1f}%)')

In [ ]:
# PnL at various edge thresholds
thresholds = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20]
print(f"{'Threshold':>10s} {'Trades':>8s} {'PnL (cents)':>12s} {'Win Rate':>10s} {'Avg Edge':>10s}")
print('-' * 55)

for thresh in thresholds:
    pnl = simulate_pnl([result], edge_threshold=thresh)
    print(f'{thresh:>10.2f} {pnl.n_trades_taken:>8d} {pnl.gross_pnl_cents:>12d} '
          f'{pnl.win_rate:>10.1%} {pnl.avg_edge_taken:>10.3f}')

In [ ]:
# Cumulative PnL over time at threshold=0.05
threshold = 0.05
cum_pnl = [0]
trade_indices = []

for i, t in enumerate(result.trades):
    if t.edge > threshold:
        # Buy YES
        if result.outcome_yes:
            cum_pnl.append(cum_pnl[-1] + (100 - t.trade_price))
        else:
            cum_pnl.append(cum_pnl[-1] - t.trade_price)
        trade_indices.append(i)
    elif t.edge < -threshold:
        # Buy NO
        if not result.outcome_yes:
            cum_pnl.append(cum_pnl[-1] + t.trade_price)
        else:
            cum_pnl.append(cum_pnl[-1] - (100 - t.trade_price))
        trade_indices.append(i)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(cum_pnl)), cum_pnl, linewidth=1)
ax.axhline(0, color='black', linestyle=':')
ax.set_xlabel('Trade number')
ax.set_ylabel('Cumulative PnL (cents)')
ax.set_title(f'Cumulative PnL — {chosen} (threshold={threshold})')
plt.tight_layout()
plt.show()

print(f'Final PnL: {cum_pnl[-1]} cents ({cum_pnl[-1]/100:.2f} dollars)')
print(f'Trades taken: {len(cum_pnl)-1}')

In [ ]:
# Brier score: model vs market for this game
model_probs = [t.model_prob_yes for t in result.trades]
market_probs = [t.market_implied for t in result.trades]
outcomes = [result.outcome_yes] * len(result.trades)

model_brier = brier_score(model_probs, outcomes)
market_brier = brier_score(market_probs, outcomes)

print(f'Model Brier:  {model_brier:.4f}')
print(f'Market Brier: {market_brier:.4f}')
print(f'Difference:   {model_brier - market_brier:.4f} ({"model worse" if model_brier > market_brier else "model better"})')